In [5]:
!pip install SPARQLWrapper pylcs

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for pylcs: filename=pylcs-0.1.1-cp311-cp311-linux_x86_64.whl size=1093486 sha256=4c11f7d2a77e76b44dcb0ded3268e3aadf04fff4e45649d2cdb411315e129f3c
  Stored in directory: /root/.cache/pip/wheels/2c/85/05/d2967d0409742801c325c5b51269c1d33d87bb90fd31d5876a
Successfully built pylcs


In [11]:
import multiprocessing

from SPARQLWrapper import SPARQLWrapper, JSON
import requests
import pylcs

import os
import pickle
import jsonpickle
import json

In [24]:
class Candidate:
    """
    This class saves candidate's information.
    Each candidate is assigned to specific entity (one-to-many).
    """

    def __init__(self, uri, label, ont_type, abstract=""):
        self.uri = uri  # from DBpedia - link to the page with details
        self.label = label  # from DBpedia - rdfs:label value
        self.ont_type = ont_type  # from DBpedia - ontology type
        self.abstract = abstract  # from DBpedia - entity abstract (optional)

class Entity(object):
    """
    Entity mention class
    """

    def __init__(self):
        self.surface_form: str
        self.ner_class: str
        self.position: (int, int)
        # self.sentence_number
        # self.sentence_text
        self.uri: str
        self.dbpedia_class: str
        self.candidates: list[Candidate]

    def set_dbp_class(self, new_class):
        self.dbpedia_class = new_class

    def set_candidates(self, candidates):
        self.candidates = candidates

    def set_uri_from_candidates(self):
        if len(self.candidates) > 0:
            self.uri = self.candidates[0].uri

    def to_html(self) -> str:
        return f"<span><a href=\"{self.uri}\">{self.surface_form}</a><sup>{self.ner_class} | {self.dbpedia_class}</sup></span>"

class TestEntity(Entity):
    def __init__(self):
        self.target_uri: str
        super().__init__()


class Text(object):
    """
    Text class for processing
    """

    def __init__(self, text: str) -> None:
        self.text: str = text
        self.entity_mentions: [Entity] = []

    def set_entity_mentions(self, entities: [Entity]):
        self.entity_mentions = entities

    def clear_entities(self):
        self.entity_mentions.clear()

    def clear_candidates(self):
        for entity in self.entity_mentions:
            entity.candidates.clear()

    def get_entity_mentions(self):
        return self.entity_mentions

    def get_plain_text(self):
        return self.text

    def get_tagged_text(self):
        """Get text with named entity mentions tagged with classes"""
        tagged_text = self.text

        for entity in self.entity_mentions[::-1]:
            entity: Entity
            tagged_text = tagged_text[
                          :entity.position[0]] + f"[{entity.surface_form}<{entity.uri}>]" + tagged_text[
                                                                                                  entity.position[1]:]
        return tagged_text

    def get_html_text(self):
        """Get text with named entity mentions linked in html"""
        tagged_text = self.text

        for entity in self.entity_mentions[::-1]:
            entity: Entity
            tagged_text = tagged_text[:entity.position[0]] + entity.to_html() + tagged_text[entity.position[1]:]
        return tagged_text


class TestText(Text):
    def __init__(self, text: str) -> None:
        self.entity_mentions: [TestEntity]
        super().__init__(text)

In [20]:

#cannot query by wikipageID
def dbpedia_lookup_query_for_url(article_name):
     # DBpedia Lookup API endpoint
        api_url = "https://lookup.dbpedia.org/api/search"

        # Parameters for the GET request
        params = {'query': article_name, 'format': 'json', 'maxResults': 1}

        # Make the GET request
        response = requests.get(api_url, params=params)

        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            # Parse the JSON data from the response
            data = response.json()["docs"][0]
            return data.get("resource",[])[0] if data.get("resource") else None
        else:
            # If the request was not successful, print an error message
            print(f"Error: {response.status_code}")
            return None

def sparql_query_for_url(sparql, wiki_id):
    """
    Queries the DBpedia SPARQL endpoint for an entity corresponding to the given Wikipedia page ID.

    Args:
        sparql (SPARQLWrapper): A configured SPARQLWrapper instance.
        wiki_id (int): The Wikipedia page ID to query for.

    Returns:
        str: The DBpedia entity URL if found, otherwise None.
    """
    query = f'''
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX dc: <http://purl.org/dc/elements/1.1/>
    PREFIX : <http://dbpedia.org/resource/>
    PREFIX dbpedia2: <http://dbpedia.org/property/>
    PREFIX dbpedia: <http://dbpedia.org/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX dbo: <http://dbpedia.org/ontology/>

    SELECT ?entity
    WHERE {{
        ?entity dbo:wikiPageID ?id .
        FILTER (?id = {wiki_id})
    }}
    '''

    try:
        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)
        print(f"Now querying for id: {wiki_id}")
        query_result = sparql.query().convert()

        # TODO: Check if query_result contains valid results
        bindings = query_result.get("results", {}).get("bindings", [])
        if bindings:
            return bindings[0]["entity"]["value"]

        # TODO: If no result is found, return None
        return None

    except Exception as e:
        # TODO: Handle exceptions gracefully (e.g., network issues, SPARQL syntax errors)
        print(f"Error querying DBpedia for ID {wiki_id}: {e}")
        return None


def getUri(article_name, title_dict, sparql):
    """
    Retrieves the DBpedia URI for a given Wikipedia article name.

    Args:
        article_name (str): The Wikipedia article title.
        title_dict (dict): A dictionary mapping article names to Wikipedia page IDs.
        sparql (SPARQLWrapper): A configured SPARQLWrapper instance.

    Returns:
        str: The DBpedia URI for the entity.
    """
    if article_name in title_dict:
        wiki_page_id = title_dict[article_name]
        target_uri = sparql_query_for_url(sparql, wiki_page_id)

        # TODO: If the SPARQL query returns None, construct a fallback URI
        if target_uri is None:
            target_uri = f"https://dbpedia.org/resource/{article_name.replace(' ', '_')}"
    else:
        target_uri = f"https://dbpedia.org/resource/{article_name.replace(' ', '_')}"

    return target_uri

def convert(filename, title_dict):
    file = open(os.path.join("datasets", filename),encoding="UTF-8")
    setcontent = file.read()
    set_list = [json.loads(line) for line in setcontent.splitlines()]

    sparql = SPARQLWrapper('https://dbpedia.org/sparql')  # initialize SPARQL Wrapper
    # Set the timeout to 3000 milliseconds
    timeout = 45  # seconds
    sparql.timeout = timeout * 1000  # milliseconds


    text = Text("")

    texts = []
    for set in set_list:
        left = (set['left_context']+" ").lstrip().replace("  "," ")
        mention = set['mention']
        right = " "+set['right_context'].replace("  "," ")
        article_name = set['output']

        main_text = (left+mention+right).strip().replace("  "," ")

        sequence_length = pylcs.lcs_sequence_length(main_text.lower().replace("."," "),text.text.lower().replace("."," "))
        overlap = sequence_length/(max(len(text.text),len(main_text)))
        sequence = pylcs.lcs_sequence_idx(text.text.lower() , main_text.lower() )
        #if text is new
        if  overlap < 0.8 : #and  0 not in sequence
            text = TestText(main_text)


            entity = TestEntity()
            entity.surface_form = mention
            entity.ner_class=""
            entity.position=(len(left),len(left)+len(mention))

            entity.target_uri = getUri(article_name,title_dict,sparql)

            text.set_entity_mentions([entity])
            texts.append(text)
        #if text is the same as last one
        else:
            offset= 0
            if -1 in sequence:
                index = sequence.index(0)
                text_prev = text.text[:index]
                text.text = text_prev+main_text
                offset = len(text_prev)#index

            entity = TestEntity()
            entity.surface_form = mention
            entity.ner_class=""
            entity.position=(len(left)+offset,len(left)+offset+len(mention))
            entity.target_uri = getUri(article_name,title_dict,sparql)

            value = text.text[entity.position[0]:entity.position[1]]
            if (mention.lower().replace("."," ").replace(","," ") != value.lower().replace("."," ")):
                print(value)

            text.get_entity_mentions().append(entity)

        #os.system('cls')
        print(filename+":",set_list.index(set)/len(set_list))

    file = open(os.path.join("output", filename.split(".")[0]+".json"),mode = "w",encoding="utf-8")
    file.write(json.dumps(jsonpickle.encode(texts)))
    file.close()


In [21]:
file = open("/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/title_dict.pickle", 'rb')
title_dict = pickle.load(file)
file.close()
title_dict = { v:k for k,v in title_dict.items()}


convert("/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl",title_dict)

# args = [(filename,title_dict) for filename in os.listdir("datasets")]

# with multiprocessing.Pool() as pool:
#     pool.starmap(convert, args)

Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8633867898885567
Now querying for id: 14579
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8634411524870889
Now querying for id: 16275
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8634955150856211
Now querying for id: 3434750
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8635498776841533
Now querying for id: 14579
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8636042402826856
Now querying for id: 212635
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.8636586028812178
Now querying for id: 3434750
/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.jsonl: 0.86371296547975
Now querying for id: 16275
/content/drive/MyDrive/Praca magiste

In [25]:
import json
from typing import List

# Assuming the classes are defined in the same script or imported

def load_texts_from_json(file_path: str) -> List[TestText]:
    with open(file_path, "r", encoding="utf-8") as file:
        raw_data = file.read()
        data = json.loads(jsonpickle.decode(raw_data))


    texts = []
    for text_obj in data:
        text = text_obj["text"]
        entity_mentions = []

        for entity_data in text_obj["entity_mentions"]:
            entity = TestEntity()
            entity.surface_form = entity_data["surface_form"]
            entity.ner_class = entity_data["ner_class"]
            entity.position = tuple(entity_data["position"]["py/tuple"])
            entity.target_uri = entity_data["target_uri"]
            entity.uri = entity.target_uri  # Setting uri from target_uri
            entity_mentions.append(entity)

        test_text = TestText(text)
        test_text.set_entity_mentions(entity_mentions)
        texts.append(test_text)

    return texts

# Example usage
file_path = "/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train.json"
texts = load_texts_from_json(file_path)

# Printing a sample output
for text in texts[:1]:  # Display first text object
    print(text.get_tagged_text())


EU rejects [German<http://dbpedia.org/resource/Germany>] call to boycott [British<http://dbpedia.org/resource/United_Kingdom>] lamb . Peter Blackburn [BRUSSELS<http://dbpedia.org/resource/Brussels>] 1996-08-22 The [European Commission<http://dbpedia.org/resource/European_Commission>] said on Thursday it disagreed with [German<http://dbpedia.org/resource/Germany>] advice to consumers to shun [British<http://dbpedia.org/resource/United_Kingdom>] lamb until scientists determine whether mad cow disease can be transmitted to sheep . [Germany<http://dbpedia.org/resource/Germany>] 's representative to the [European Union<http://dbpedia.org/resource/European_Union>] 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than [Britain<http://dbpedia.org/resource/United_Kingdom>] until the scientific advice was clearer . " We do n't support any such recommendation because we do n't see any grounds for it , " the [Commission<http://dbpedia.o

In [26]:
import json

def convert_texts_to_new_format(texts, output_file):
    formatted_data = []

    for text_obj in texts:
        formatted_text = {
            "text": text_obj.text,
            "entity_mentions": []
        }

        for entity in text_obj.get_entity_mentions():
            entity_data = {
                "surface_form": entity.surface_form,
                "start_position": entity.position[0],
                "end_position": entity.position[1],
                "target_uri": entity.target_uri
            }
            formatted_text["entity_mentions"].append(entity_data)

        formatted_data.append(formatted_text)

    # Save to JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(formatted_data, f, ensure_ascii=False, indent=2)

# Example usage
output_file = "/content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train_converted.json"
convert_texts_to_new_format(texts, output_file)

print(f"Converted data saved to {output_file}")


Converted data saved to /content/drive/MyDrive/Praca magisterska/Google colab - notebooks/aida_train_converted.json
